# Day 4

## Tokenizing with code

In [7]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi I'm Batman and I like to enforce justice.")

In [8]:
tokens

[12194, 5477, 54437, 326, 357, 1299, 316, 57352, 20604, 13]

In [9]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
5477 =  I'm
54437 =  Batman
326 =  and
357 =  I
1299 =  like
316 =  to
57352 =  enforce
20604 =  justice
13 = .


In [10]:
encoding.decode([326])

' and'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [13]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [16]:
from openai import AsyncOpenAI

MODEL = "gemini-3.1-flash-lite"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = AsyncOpenAI(api_key=api_key, base_url=GEMINI_BASE_URL)

### A message to OpenAI is a list of dicts

In [15]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Batman!"}
    ]

In [17]:
from IPython.display import Markdown, display

response = await gemini.chat.completions.create(model=MODEL, messages=messages)
display(Markdown(response.choices[0].message.content))

It’s an honor, Batman. Gotham is in good hands. 

How can I help you today? Is there a case you’re working on, or do you need assistance with the Batcomputer?

### OK let's now ask a follow-up question

In [18]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Do you know my secret identity?"}
    ]

In [19]:
response = await gemini.chat.completions.create(model=MODEL, messages=messages)
display(Markdown(response.choices[0].message.content))

As an AI, I don't have access to your personal life, your webcam, or any private files unless you specifically share them with me in this chat. 

So, if you *do* have a secret identity—whether you're a mild-mannered reporter, a billionaire philanthropist, or just someone who enjoys a good disguise—your secret is safe with me! 

Are you currently "on duty," or is this just a hypothetical question?

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [20]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Batman!"},
    {"role": "assistant", "content": "Hi Batman! How can I assist you today?"},
    {"role": "user", "content": "What's my secret identity?"}
    ]

In [22]:
response = await gemini.chat.completions.create(model=MODEL, messages=messages)
display(Markdown(response.choices[0].message.content))

That’s a question for the shadows, wouldn’t you say? 

But between us—I know the secret stays safe in the Batcave. Shall we focus on crime-fighting, tech upgrades, or perhaps something else entirely?

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

